In [7]:
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from tavily import TavilyClient 

load_dotenv(override=True)

api_key = os.getenv("LLM_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

if api_key and tavily_api_key:
    print("Api keys loaded successfully")
else:
    print("Api keys not found")



Api keys loaded successfully


In [8]:
print("LANGSMITH_ENDPOINT =", os.getenv("LANGSMITH_ENDPOINT"))

LANGSMITH_ENDPOINT = https://api.smith.langchain.com


In [3]:
# LangChain tools are wrappers around external capabilities
# that an LLM (or agent) can call while reasoning.

# A tool is usually a normal Python function,
# but it is exposed to the LLM in a structured way.

# Tools allow an LLM to:
# - Fetch external data (web search, database queries)
# - Perform actions (API calls, calculations)
# - Use real-world capabilities instead of guessing

# Without tools:
# - The LLM can only generate text from its training data
# - It cannot access live data or run code

# With tools:
# - The LLM can decide when it needs help
# - Call a specific tool
# - Use the tool's output to continue reasoning

# In LangChain, tools are typically created using the @tool decorator.

# Example:
# @tool
# def get_weather(city: str) -> str:
#     return "Weather info"

# The decorator:
# - Registers the function as a tool
# - Extracts the function name and docstring
# - Creates a schema so the LLM knows how to call it

# Internally, each tool has:
# - A name (used by the agent)
# - A description (so the LLM knows when to use it)
# - An input schema (arguments)
# - An output type

# During agent execution:
# 1. The LLM analyzes the user question
# 2. Decides whether a tool is needed
# 3. Selects the appropriate tool
# 4. Passes arguments to the tool
# 5. Receives the tool output
# 6. Uses the output to generate the final answer

# Tools are commonly used for:
# - Web search (e.g., Tavily)
# - Database access
# - File system operations
# - API calls
# - Math or code execution
# - Retrieval-Augmented Generation (RAG)

# Tools help reduce hallucinations by grounding responses
# in real data or deterministic operations.

# Mental model:
# - LLM = brain (reasoning)
# - Tool = hands (actions)
# - LangChain = coordinator

# In short:
# LangChain tools let LLMs act, not just talk.


In [14]:
# tavily.search(query=...) performs a live web search
# and returns results optimized for LLM consumption.

# When this function is called, Tavily:
# 1. Interprets the query intent (not just keyword matching)
# 2. Expands/refines the query for factual accuracy
# 3. Searches the live internet for relevant sources
# 4. Fetches high-signal pages (docs, blogs, news, GitHub, etc.)
# 5. Removes ads, HTML, navigation bars, and noise
# 6. Extracts only the meaningful textual content
# 7. Ranks results using semantic relevance (embeddings)
# 8. Filters out low-quality or redundant information
# 9. Summarizes content in a query-aware, factual way
# 10. Compresses the information to save tokens
# 11. Structures the output so an LLM can read it easily

# The returned data typically includes:
# - Concise summaries of relevant content
# - Source URLs for grounding and traceability
# - Relevance scores or rankings
# - Clean text (not raw HTML)

# This allows an LLM or agent to:
# - Access up-to-date information
# - Reduce hallucinations
# - Use web knowledge during reasoning
# - Avoid manual scraping or crawling logic

# Mental model:
# tavily.search() = "Search the web, read it, clean it, summarize it, and hand it to the LLM"

# In short:
# tavily.search(query=...) integrates live web search directly into an LLM workflow.


In [11]:
tavily = TavilyClient()

@tool
def search(search_query:str) -> str:
    """ 
    Tool that searches over internet
    Args:
        query: The query to search for 
    Returns:
        The search results 
    """
    print(f"Searching for {search_query}")

    return tavily.search(query=search_query)
    

In [12]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tools = [search]
agent = create_agent(llm, tools)

In [13]:
result = agent.invoke({"messages" : HumanMessage(content="What is the weather in Tokyo?")})
print(result)

Searching for current weather in Tokyo
{'messages': [HumanMessage(content='What is the weather in Tokyo?', additional_kwargs={}, response_metadata={}, id='f57ee544-24d5-4c19-8796-cc423abc95d9'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 68, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1590f93f9d', 'id': 'chatcmpl-D45THOLaRPykMl6lUstffp5EQfnag', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c1446-e033-7082-aec8-c7256b2c9e50-0', tool_calls=[{'name': 'search', 'args': {'search_query': 'current weather in Tokyo'}, 'id': 'call_peFrx9OIZYQCeMFFVqEiW5MD', 'type': 'tool_call'}], invalid

In [15]:
# HumanMessage is a LangChain class used to represent a message
# that comes from a human (the user) in a chat-based conversation.

# It is part of LangChain's message abstraction layer,
# which standardizes how messages are passed to chat models.

# HumanMessage is typically used when:
# - Sending user input to a chat model
# - Manually constructing conversation history
# - Working with multi-turn conversations
# - Interacting directly with chat model APIs

# Instead of passing raw strings, LangChain uses message objects
# to clearly indicate who said what in a conversation.

# Example conceptual roles in a chat:
# - HumanMessage   → user input
# - AIMessage      → model response
# - SystemMessage  → system instructions

# Using HumanMessage helps LangChain:
# - Track conversation state
# - Maintain role separation
# - Format prompts correctly for chat models
# - Support multiple LLM providers consistently

# HumanMessage usually contains:
# - content: the text the user typed
# - optional metadata (like name or additional fields)

# Mental model:
# HumanMessage = "this text was written by the user"

# In short:
# HumanMessage is used to wrap user input in a structured,
# role-aware format before sending it to a chat model.
